In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# Partie 1 : le fichier des données financières

In [ ]:
# en millions de dirhams (M MAD) /  source : rapports financiers annuels Managem, comptes consolidés IFRS

In [ ]:
Business_plan= pd.DataFrame({
    "annee":          [2023,    2024,    2025],
    "CA":             [7507.6,  8859.4,  13693.6],
    "EBIT":           [1001.6,  1287.7,  3992.8],
    "DA":             [1397.0,  1368.9,  2012.4],
    "RN_Part_Grpe":   [514.4,   619.8,   3002.0],
    "EBT":            [623.0,   973.5,   3798.0],
    "impot":          [-13.8,   198.5,   389.1],
    "capex":          [2755.9,  7254.0,  5950.0],
    "var_BFR":        [-73.4,   -476.0,  507.0],
    "dette_fin":      [9595.6,  12399.5, 15323.5],
    "tresorerie":     [2740.5,  2589.1,  4390.3],
    "minoritaires":   [662.7,   730.9,   1051.3],}).set_index("annee")

Business_plan["RN_Part_Grpe"]=Business_plan["RN_Part_Grpe"]-Business_plan["impot"]-Business_plan["minoritaires"]
Business_plan 

In [ ]:
bsplan = pd.DataFrame({
    "annee":                [2023,    2024,    2025],
    "CA":                    [7507.6,  8859.4,  13693.6],
    "DA":                    [1397.0,  1368.9,  2012.4],  
    "EBIT":                  [1001.6,  1287.7,  3992.8],   # résultat des activités opérationnelles
    "resultat_financier":    [-378.6,  -314.3,  -194.7],
    "impot":                 [-13.8,   198.5,   389.1],    # impôts sur les bénéfices + impôts différés (charge)
    "mise_en_equivalence":   [-95.8,   12.5,    61.4],
    "minoritaires_resultat": [26.6,    167.6,   468.4],    # part des minoritaires dans le résultat
     "dette_non_courante":   [6832.1, 7951.6, 10544.2],
    "derives_non_courants": [244.7,  560.7,  1707.9],
    "dette_courante":       [3010.0, 5028.3, 6487.2],
    "derives_courants":     [1.8,    19.7,   0.0],
    "tresorerie":           [2740.5, 2589.1, 4390.3],
    "minoritaires":         [662.7,  730.9,  1051.3],
     "capex":                [2755.9, 7254.0, 5950.0],    # acquisitions d'immobilisations
    "var_BFR":               [-73.4, -476.0, 507.0],
    "capitaux_propres":     [8227.7, 11418.7, 12529.6]          
}).set_index("annee")

bsplan["EBITDA"] = bsplan["EBIT"] + bsplan["DA"]
bsplan["EBT"] = bsplan["EBIT"] + bsplan["resultat_financier"]
bsplan["R.net"] = bsplan["EBT"] - bsplan["impot"] + bsplan["mise_en_equivalence"]
bsplan["RN.PartGrp"] = bsplan["R.net"] - bsplan["minoritaires_resultat"]
bsplan["dette_nette"] = bsplan["dette_non_courante"]+bsplan["dette_courante"]-bsplan["derives_non_courants"]-bsplan["derives_courants"] - bsplan["tresorerie"]
# on remet les colonnes dans l'ordre du compte de résultat

bsplan = bsplan[["CA", "EBITDA", "DA", "EBIT", "resultat_financier", "EBT", "impot",
         "mise_en_equivalence", "R.net", "minoritaires_resultat", "RN.PartGrp","dette_nette","capex","var_BFR" ]]

bsplan





In [ ]:
nb_actions = 118_646_760          # après la division par dix du 27 juillet 2026
cours = 1469                 
capitalisation = cours * nb_actions / 1e6     #1e6 convertit  en millions de MAD     

In [ ]:
f = bsplan.copy()
f["croissance_CA"] = f["CA"].pct_change()
f["marge_EBITDA"] = f["EBITDA"] / f["CA"]
f["marge_EBIT"] = f["EBIT"] / f["CA"]
f["taux_IS"] = f["impot"] / f["EBT"]
f["capex_CA"] = f["capex"] / f["CA"]

# free cash flow pour l'entreprise (FCFF) : l'argent disponible pour les actionnaires ET les prêteurs
f["FCFF"] = f["EBIT"] * (1 - f["taux_IS"]) + f["DA"] - f["capex"] - f["var_BFR"]

f["levier"] = bsplan["dette_nette"] / bsplan["EBITDA"]      # dette nette / EBITDA

f.round(3).T

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

ax.bar(f.index - 0.2, f["CA"], width=0.4, label="Chiffre d'affaires")
ax.bar(f.index + 0.2, f["EBITDA"], width=0.4, label="EBITDA")
ax.set_xticks(f.index)
ax.set_title("Managem : chiffre d'affaires, EBITDA et levier")
ax.set_ylabel("Millions de MAD")

ax2 = ax.twinx()                                   # deuxième axe, à droite
ax2.plot(f.index, f["levier"], color="black", marker="o", label="Levier (dette nette / EBITDA)")
ax2.set_ylabel("Levier (x)")
ax2.set_ylim(0, f["levier"].max() * 1.3)

h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, loc="upper left")

plt.tight_layout()
plt.show()

In [ ]:
# 1- Entre 2024 et 2025, la marge d’EBITDA a fortement progressé, de 30 % à 44 %, et le levier a été divisé par deux, de 3,7x à 1,8x. Deux effets se combinent : l’entrée en production de Boto (or) et de Tizert (cuivre) au quatrième trimestre 2025, et la hausse des prix de l’or, de l’argent et du cuivre. Comme une grande partie des coûts d’une mine sont fixes, la hausse des prix se retrouve presque entièrement dans l’EBITDA : c’est le levier opérationnel. La dette nette a pourtant continué d’augmenter, de 9,8 à 10,9 milliards de dirhams, mais l’EBITDA a plus que doublé, d’où la forte baisse du levier. En 2024, au contraire, le levier avait atteint 3,7x : la dette finançait la construction des mines, qui ne produisaient pas encore
2- Oui, le FCFF de 2024 est très négatif, à −4,4 milliards de dirhams, à cause d’investissements record de 7,3 milliards pour construire Boto et Tizert. Ce n’est pas inquiétant en soi : c’est le profil normal d’une entreprise minière en phase de développement, qui investit plusieurs années avant de produire. Ce qui compte, c’est que ces investissements soient financés sans fragiliser l’entreprise : Managem a combiné une augmentation de capital d’environ 3 milliards et de nouveaux emprunts. Le FCFF reste encore légèrement négatif en 2025, car les mines n’ont produit que quelques mois. Il devrait devenir nettement positif en 2026, première année pleine de production : on le voit déjà au premier semestre 2026, avec un EBITDA de 6,2 milliards et des investissements retombés à 1,7 milliard
3- Non, ce serait risqué. Doubler le S1 2026 donnerait un chiffre d’affaires d’environ 23,5 milliards et un EBITDA d’environ 12,4 milliards, mais plusieurs raisons invitent à la prudence. D’abord, Managem annonce elle-même que le second semestre sera une “étape de consolidation”. Ensuite, la production a déjà baissé au premier semestre à Imiter (argent) et à Tri-K (or), à cause de difficultés opérationnelles. Enfin, la marge de 53 % dépend des prix très élevés des métaux, qui peuvent baisser à tout moment. Pour une entreprise cyclique, un semestre exceptionnel ne se prolonge pas mécaniquement : il vaut mieux construire plusieurs scénarios de prix et de volumes

In [ ]:
f["capitaux_propres"] = [8227.7, 11418.7, 12529.6]
x = np.arange(len(f.index))      # positions 0, 1, 2 pour les 3 années
largeur = 0.25

fig, ax = plt.subplots(figsize=(9, 5))

# les montants, en barres
ax.bar(x - largeur/2 ,f["capitaux_propres"], largeur, label="Capitaux propres", color="seagreen")
ax.bar(x + largeur/2 ,f["dette_nette"],      largeur, label="Dette nette", color="steelblue")
ax.set_xticks(x)
ax.set_xticklabels(f.index)
ax.set_ylabel("Millions de MAD")
ax.set_ylim(0, f[["capitaux_propres", "dette_nette"]].max().max() * 1.4)
ax.set_title("Managem : structure financière et levier")

# le levier, en courbe sur un deuxième axe
ax2 = ax.twinx()
ax2.plot(x, f["levier"], color="black", marker="o", lw=2, label="Levier (dette nette / EBITDA)")
for xi, v in zip(x, f["levier"]):
    ax2.annotate(f"{v:.1f}x", (xi, v), textcoords="offset points", xytext=(0, 8), ha="center")
ax2.set_ylabel("Dette nette / EBITDA (x)")
ax2.set_ylim(0, f["levier"].max() * 1.3)

# une seule légende pour les deux axes
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, loc="upper left")

plt.tight_layout()
plt.show()

# Partie 2: Valorisation par Multiples Boursiers 

In [ ]:
# L'idée est simple : si des entreprises semblables valent en moyenne 6 fois leur EBITDA, Managem devrait valoir à peu près 6 fois le sien. C'est la méthode la plus utilisée en pratique, car elle reflète ce que le marché paie aujourd'hui.

In [ ]:
import yfinance as yf

comparables = {
    "Endeavour Mining": "EDV.L",   # Or, Afrique de l'Ouest
    "Perseus Mining":   "PRU.AX",  # Or, Afrique de l'Ouest
    "B2Gold":           "BTG",     # Or, Afrique et Amériques
    "IAMGOLD":          "IAG",     # Or, ancien propriétaire de Boto
    "Fresnillo":        "FRES.L",  # argent et Or
    "Lundin Mining":    "LUN.TO",  # cuivre
    "Boliden":          "BOL.ST", }  # cuivre et zinc 
lignes = []                             #crée liste vide 
for nom, ticker in comparables.items():
    info = yf.Ticker(ticker).info
    lignes.append({"Société": nom,
                   "VE/EBITDA": info.get("enterpriseToEbitda"),
                   "PER": info.get("trailingPE"),
                   "Devise": info.get("currency"), "Cours": info.get("currentPrice")})

multiples = pd.DataFrame(lignes).set_index("Société")
multiples

In [ ]:
#mutiple médian 
VE_EBITDA_median = multiples["VE/EBITDA"].median()
PER_median = multiples["PER"].median()

#parameter de Mangem 
EBITDA_2025 = f.loc[2025, "EBITDA"]
RN_PartGrp_2025 = f.loc[2025, "RN.PartGrp"]
dette_nette = f.loc[2025, "dette_nette"] 
minoritaires = f.loc[2025, "minoritaires_resultat"]

# méthode VE/EBITDA
VE_MNG_1 = VE_EBITDA_median * EBITDA_2025
Valeur_actions_MNG = VE_MNG_1 - dette_nette - minoritaires 
prix_Action_MNG= Valeur_actions_MNG * 1e6 / nb_actions

# méthode PER
EqV_MNG_2= PER_median * RN_PartGrp_2025
VE_MNG_2=EqV_MNG_2+dette_nette
prix_per = PER_median * RN_PartGrp_2025 * 1e6 / nb_actions

print(f"Prix par action, VE/EBITDA : {prix_Action_MNG:,.0f} MAD")
print(f"Prix par action, PER       : {prix_per:,.0f} MAD")
print(f"Cours actuel               : {cours:,.0f} MAD")
print (f"EV/EBITDA de Mangemn , VE/EBITDA : {VE_MNG_1/EBITDA_2025:,.0f}")

# Partie 2 : DCF 

In [ ]:
#calcul WACC 

In [ ]:

rf = 0.0378            # bons du Trésor marocains à 10 ans
risk premium = 0.042  # prime de marché mature (Damodaran)
kD = 0.0673            # coût de la dette avant impôt
t = 0.02             # taux d'impôt (effectif ou légal)

kE = rf + beta * (prime_marche )

#calcul beta : 
mng = pd.read_csv("MNG.csv", index_col=0, parse_dates=True)["Cours"]
masi = pd.read_csv("MASI.csv", index_col=0, parse_dates=True)["Cours"]
print(mng.head()) 
donnees = pd.concat([mng, masi], axis=1, join="inner", keys=["MNG", "MASI"])
rend = donnees.pct_change().dropna()
beta = rend.cov().loc["MNG", "MASI"] / rend["MASI"].var()
print(f"Bêta de Managem : {beta:.2f}")


E = capitalisation
D = dette_nette
WACC = E / (E + D) * kE + D / (E + D) * kD * (1 - t)

print(f"Coût des fonds propres : {kE:.2%}")
print(f"WACC : {WACC:.2%}")